In [1]:
import pandas as pd
import numpy as np
import pickle
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 1. Load Data
train_df = pd.read_csv('data/processed/train_interactions.csv')
movies_meta = pd.read_csv('data/processed/movie_metadata.csv')

with open('data/processed/user2idx.pkl', 'rb') as f:
    user2idx = pickle.load(f)
with open('data/processed/movie2idx.pkl', 'rb') as f:
    movie2idx = pickle.load(f)
with open('data/processed/movie_embeddings.pkl', 'rb') as f:
    movie_embeddings = pickle.load(f)

num_users = len(user2idx)
num_movies = len(movie2idx)
print(f"Users: {num_users}, Movies: {num_movies}")

# Convert embeddings dict to a contiguous tensor for the Item Tower
emb_dim_text = 384
text_emb_matrix = torch.zeros((num_movies, emb_dim_text))
for m_idx, emb in movie_embeddings.items():
    text_emb_matrix[m_idx] = torch.tensor(emb)


Using device: cuda
Users: 102128, Movies: 10326


In [2]:
class HistoryBPRDataset(Dataset):
    def __init__(self, df, num_movies, max_history_length=50):
        self.users = torch.tensor(df['user_idx'].values, dtype=torch.long)
        self.pos_items = torch.tensor(df['movie_idx'].values, dtype=torch.long)
        self.num_movies = num_movies
        self.max_len = max_history_length
        
        # Group histories and keep them in order (chronological if dataframe is sorted)
        print("Building user histories for feature-based tower...")
        grouped = df.groupby('user_idx')['movie_idx'].apply(list)
        self.user_history_list = grouped.to_dict()
        self.user_history_set = grouped.apply(set).to_dict() # For fast negative sampling

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        u = self.users[idx].item()
        pos_i = self.pos_items[idx].item()
        
        # --- NEGATIVE SAMPLING ---
        neg_i = np.random.randint(0, self.num_movies)
        while neg_i in self.user_history_set[u]:
            neg_i = np.random.randint(0, self.num_movies)
            
        # --- PADDING & MASKING ---
        # Get user history (excluding the current positive item to prevent data leakage)
        full_history = [m for m in self.user_history_list[u] if m != pos_i]
        
        # Truncate if too long (keep most recent)
        if len(full_history) > self.max_len:
            history = full_history[-self.max_len:]
        else:
            history = full_history
            
        # Create padded array and mask
        hist_array = np.zeros(self.max_len, dtype=np.int64)
        mask_array = np.zeros(self.max_len, dtype=np.float32)
        
        # Fill the arrays
        length = len(history)
        if length > 0:
            hist_array[:length] = history
            mask_array[:length] = 1.0 # 1 means real data, 0 means padding
            
        return (
            torch.tensor(hist_array), 
            torch.tensor(mask_array), 
            torch.tensor(pos_i), 
            torch.tensor(neg_i)
        )

# Initialize new dataset
train_dataset = HistoryBPRDataset(train_df, num_movies, max_history_length=50)


print("Initializing Dataset and DataLoader...")

train_loader = DataLoader(train_dataset, batch_size=2048, shuffle=True, num_workers=2)


Building user histories for feature-based tower...
Initializing Dataset and DataLoader...


In [3]:
class HistoryBasedTwoTower(nn.Module):
    def __init__(self, num_movies, text_emb_matrix, latent_dim=64):
        super().__init__()
        # 1. Shared Item Features
        self.text_embeddings = nn.Embedding.from_pretrained(text_emb_matrix, freeze=True)
        text_dim = text_emb_matrix.shape[1] 
        self.item_emb = nn.Embedding(num_movies, latent_dim)
        
        # 2. Projection Network (Used by both items and histories)
        self.proj = nn.Sequential(
            nn.Linear(latent_dim + text_dim, 256), nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 64)
        )
        
    def get_item_rep(self, item_ids):
        # Generates a vector for a given movie ID (or batch of movie IDs)
        i_cf = self.item_emb(item_ids)
        i_text = self.text_embeddings(item_ids)
        i_concat = torch.cat([i_cf, i_text], dim=-1)
        return self.proj(i_concat)

    def get_user_rep(self, history_ids, history_mask):
        # 1. Get vectors for all $N$ movies in the history
        # history_ids shape: (Batch, Max_Len) -> output: (Batch, Max_Len, 64)
        hist_reps = self.get_item_rep(history_ids) 
        
        # 2. Smart Aggregation (Masked Average Pooling)
        mask = history_mask.unsqueeze(-1) # Shape: (Batch, Max_Len, 1)
        
        # Multiply by mask to zero out the padding embeddings
        sum_reps = (hist_reps * mask).sum(dim=1) # Sum real movies
        
        # Count how many real movies there are (avoid dividing by zero)
        lengths = mask.sum(dim=1).clamp(min=1e-9) 
        
        # Divide by actual number of watched movies, not max length
        user_rep = sum_reps / lengths 
        return user_rep

    def forward(self, history_ids, history_mask, pos_item_ids, neg_item_ids):
        u_rep = self.get_user_rep(history_ids, history_mask)
        pos_rep = self.get_item_rep(pos_item_ids)
        neg_rep = self.get_item_rep(neg_item_ids)
        
        # Dot product
        pos_scores = (u_rep * pos_rep).sum(dim=1)
        neg_scores = (u_rep * neg_rep).sum(dim=1)
        return pos_scores, neg_scores

model = HistoryBasedTwoTower(num_movies, text_emb_matrix).to(device)


optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)


In [5]:
# Train/Validation Split and Training Loop
from torch.utils.data import random_split
import torch.nn.functional as F
from tqdm import tqdm

# Split the dataset (90% train, 10% validation)
train_size = int(0.9 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_subset, val_subset = random_split(train_dataset, [train_size, val_size])

# Create DataLoaders for both splits (try with num_workers=0 first to debug)
train_loader = DataLoader(train_subset, batch_size=2048, shuffle=True, num_workers=0)
val_loader = DataLoader(val_subset, batch_size=2048, shuffle=False, num_workers=0)

# Ensure model is on the correct device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f"Using device: {device}")

# Training Loop with BPR Loss and Early Stopping
epochs = 30
patience = 3
best_val_loss = float('inf')
patience_counter = 0

for epoch in range(epochs):
    # --- Training Phase ---
    model.train()
    total_train_loss = 0
    train_progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
    
    for hist_ids, hist_masks, pos_items, neg_items in train_progress:
        hist_ids, hist_masks = hist_ids.to(device), hist_masks.to(device)
        pos_items, neg_items = pos_items.to(device), neg_items.to(device)
        
        optimizer.zero_grad()
        pos_scores, neg_scores = model(hist_ids, hist_masks, pos_items, neg_items)
        
        # BPR Loss: -log(sigmoid(pos_scores - neg_scores))
        loss = -F.logsigmoid(pos_scores - neg_scores).mean()
        loss.backward()
        optimizer.step()
        
        total_train_loss += loss.item()
        train_progress.set_postfix({'loss': loss.item()})
        
    avg_train_loss = total_train_loss / len(train_loader)
    
    # --- Validation Phase ---
    model.eval()
    total_val_loss = 0
    val_progress = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]")
    
    with torch.no_grad():
        for hist_ids, hist_masks, pos_items, neg_items in val_progress:
            hist_ids, hist_masks = hist_ids.to(device), hist_masks.to(device)
            pos_items, neg_items = pos_items.to(device), neg_items.to(device)
            
    

            
            loss = -F.logsigmoid(pos_scores - neg_scores).mean()
            total_val_loss += loss.item()
            
    avg_val_loss = total_val_loss / len(val_loader)
    print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    
    # --- Early Stopping ---
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), 'best_hybrid_model.pth')
        print("  -> Model improved. Saved 'best_hybrid_model.pth'")
    else:
        patience_counter += 1
        print(f"  -> No improvement. Patience: {patience_counter}/{patience}")
        if patience_counter >= patience:
            print("Early stopping triggered!")
            break

# Load the best model weights back in before moving to inference
model.load_state_dict(torch.load('best_hybrid_model.pth'))
print("Training complete. Best model loaded.")


Using device: cuda


Epoch 1/30 [Train]: 100% 3923/3923 [11:27<00:00,  5.71it/s, loss=0.158]
Epoch 1/30 [Val]: 100% 436/436 [01:16<00:00,  5.69it/s]


Epoch 1 | Train Loss: 0.2075 | Val Loss: 0.1578
  -> Model improved. Saved 'best_hybrid_model.pth'


Epoch 2/30 [Train]: 100% 3923/3923 [13:34<00:00,  4.82it/s, loss=0.173]
Epoch 2/30 [Val]: 100% 436/436 [01:08<00:00,  6.40it/s]


Epoch 2 | Train Loss: 0.1524 | Val Loss: 0.1735
  -> No improvement. Patience: 1/3


...

Epoch 5 | Train Loss: 0.1216 | Val Loss: 0.1162
  -> No improvement. Patience: 1/3


Epoch 6/30 [Train]: 100% 3923/3923 [13:35<00:00,  4.81it/s, loss=0.125] 
Epoch 6/30 [Val]: 100% 436/436 [01:04<00:00,  6.72it/s]


Epoch 6 | Train Loss: 0.1190 | Val Loss: 0.1245
  -> No improvement. Patience: 2/3


Epoch 7/30 [Train]: 100% 3923/3923 [13:42<00:00,  4.77it/s, loss=0.122] 
Epoch 7/30 [Val]: 100% 436/436 [01:18<00:00,  5.57it/s]

Epoch 7 | Train Loss: 0.1170 | Val Loss: 0.1223
  -> No improvement. Patience: 3/3
Early stopping triggered!
Training complete. Best model loaded.


In [6]:
import os
import torch

print("Creating inference bundle...")
model.eval()
model.cpu() # Move model to CPU before grabbing state_dict

# 1. Clean up previous runs (Drop rating_count if it already exists)
if 'rating_count' in movies_meta.columns:
    movies_meta = movies_meta.drop(columns=['rating_count'])

# Add a rating_count to metadata for the get_popular_movies() function
rating_counts = train_df['movie_idx'].value_counts().reset_index()
rating_counts.columns = ['movie_idx', 'rating_count']
movies_meta = movies_meta.merge(rating_counts, on='movie_idx', how='left').fillna(0)
movies_meta['rating_count'] = movies_meta['rating_count'].astype(int)

# 2. Standardize columns safely (only rename if the old names still exist)
rename_mapping = {
    'movieId': 'MovieID',
    'title': 'Title_Clean',
    'genres': 'Genres'
}
movies_meta = movies_meta.rename(columns={k: v for k, v in rename_mapping.items() if k in movies_meta.columns})

# Extract Year (assuming format "Title (Year)")
movies_meta['Year'] = movies_meta['Title_Clean'].str.extract(r'\((\d{4})\)')

idx_to_movie_id = {idx: mid for mid, idx in movie2idx.items()}

# We save everything into one file for the Django backend
bundle = {
    'model_state_dict': model.state_dict(),
    'model_config': {
        'num_users': num_users,
        'num_movies': num_movies,
        'latent_dim': 64
    },
    'text_emb_matrix': text_emb_matrix.cpu().numpy(), # Ensure tensor is on CPU
    'movies_meta': movies_meta,
    'movie_id_to_idx': movie2idx,
    'idx_to_movie_id': idx_to_movie_id
}

os.makedirs('bundle', exist_ok=True)
bundle_path = 'bundle/hybrid_recommender_bundle.pkl'

# Use torch.save instead of standard pickle
torch.save(bundle, bundle_path) 

print(f"Success. Bundle saved to {bundle_path}.")


Creating inference bundle...
Success. Bundle saved to bundle/hybrid_recommender_bundle.pkl.
